<a href="https://colab.research.google.com/github/gaben83/A-Geometric-Parametrization-of-Flavor-from-an-S1-Z2-Compact-Dimension/blob/main/New%20fit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
from scipy.optimize import minimize
import pandas as pd

M_EXP = np.array([2.16, 1270, 173000, 4.67, 93, 4180], dtype=float)
V_EXP = np.array([0.225, 0.041, 0.0037], dtype=float)
M_NAMES = ["mu", "mc", "mt", "md", "ms", "mb"]
V_NAMES = ["Vus", "Vcb", "Vub"]

def get_yukawa_matrix(params, q_type='up'):
    a0, phi_val, sigma, w, theta_w, eps, du, dd, q, log_ku, log_kd = params
    ku, kd = np.exp(log_ku), np.exp(log_kd)
    theta = np.array([0.0, 2*np.pi/3, 4*np.pi/3])
    R = np.sqrt(2) * a0

    if q_type == 'up':
        theta_L, theta_R, kappa = theta, theta + du, ku
    else:
        theta_L, theta_R, kappa = theta + eps, theta + eps + dd, kd

    Y = np.zeros((3, 3), dtype=complex)
    for i in range(3):
        for j in range(3):
            thL, thR = theta_L[i], theta_R[j]
            theta_avg = 0.5 * (thL + thR)
            theta_diff = thL - thR
            W_f = np.exp(-w * (1 - np.cos(thL - theta_w))) * np.exp(-w * (1 - np.cos(thR - theta_w)))
            phi_t = a0 + R * np.cos(theta_avg + phi_val)
            overlap = np.exp(-(theta_diff**2) / (4 * sigma**2))
            phase = np.exp(1j * q * theta_diff)
            Y[i, j] = kappa * W_f * phi_t * overlap * phase
    return Y

def spectrum_and_ckm(params):
    Yu = get_yukawa_matrix(params, 'up')
    Yd = get_yukawa_matrix(params, 'down')
    mu = np.sort(np.linalg.svd(Yu, compute_uv=False))
    md = np.sort(np.linalg.svd(Yd, compute_uv=False))
    Uu, _, _ = np.linalg.svd(Yu)
    Ud, _, _ = np.linalg.svd(Yd)
    V = np.abs(np.conj(Uu.T) @ Ud)
    v = np.array([V[0,1], V[1,2], V[0,2]])
    return mu, md, v, V

def objective(params):
    a0, phi_val, sigma, w, theta_w, eps, du, dd, q, log_ku, log_kd = params
    if a0 <= 0 or sigma < 0.1 or w < 0 or not (-2*np.pi <= q <= 2*np.pi):
        return 1e12
    if not (0 <= eps <= 1 and 0 <= du <= 1 and 0 <= dd <= 1):
        return 1e12
    try:
        mu, md, v, _ = spectrum_and_ckm(params)
        m_mod = np.concatenate([mu, md])
        chi2_m = 5.0 * np.sum((np.log(m_mod / M_EXP))**2)
        chi2_v = np.sum(((v - V_EXP) / np.array([0.01, 0.01, 0.003]))**2)
        penalty = 0.01 * q**2 + 0.01 * np.exp(log_ku - 10)**2 + 0.01 * np.exp(log_kd - 7)**2
        return chi2_m + chi2_v + penalty
    except Exception:
        return 1e12

def random_initial(seed=None):
    rng = np.random.default_rng(seed)
    return np.array([
        rng.uniform(5, 25),
        rng.uniform(-np.pi, np.pi),
        rng.uniform(0.2, 1.0),
        rng.uniform(0, 5),
        rng.uniform(0, 2*np.pi),
        rng.uniform(0, 0.5),
        rng.uniform(0, 0.5),
        rng.uniform(0, 0.5),
        rng.uniform(-1, 1),
        rng.uniform(8, 11),
        rng.uniform(5, 8)
    ], dtype=float)

def fit_once(seed=None, maxiter=2000):
    x0 = random_initial(seed)
    bounds = [
        (1e-3, 200),
        (-np.pi, np.pi),
        (0.1, 2.0),
        (0.0, 10.0),
        (0.0, 2*np.pi),
        (0.0, 1.0),
        (0.0, 1.0),
        (0.0, 1.0),
        (-2*np.pi, 2*np.pi),
        (6.0, 12.0),
        (4.0, 12.0)
    ]
    res = minimize(objective, x0, method='L-BFGS-B', bounds=bounds, options={'maxiter': maxiter})
    return res

def summarize(res):
    p = res.x
    mu, md, v, V = spectrum_and_ckm(p)
    out = pd.DataFrame({
        'observable': M_NAMES + V_NAMES,
        'model': np.concatenate([mu, md, v]),
        'exp': np.concatenate([M_EXP, V_EXP])
    })
    out['rel_err'] = (out['model'] - out['exp']) / out['exp']
    return out, V

N_STARTS = 40
results = []
for s in range(N_STARTS):
    res = fit_once(seed=s)
    results.append(res)
    print(f"start {s:02d}  chi2={res.fun:.4f}")

results = sorted(results, key=lambda r: r.fun)
best = results[0]
summary, Vbest = summarize(best)

print("BEST CHI2:", best.fun)
print(pd.Series(best.x, index=['a0','phi','sigma','w','theta_w','eps','du','dd','q','log_ku','log_kd']))
print(summary)
print("CKM matrix:\n", Vbest)


start 00  chi2=6022.8017
start 01  chi2=22.8188
start 02  chi2=1.4749
start 03  chi2=22.9751
start 04  chi2=84.6948
start 05  chi2=509.8706
start 06  chi2=520.4911
start 07  chi2=0.4181
start 08  chi2=57.3749
start 09  chi2=18.3300
start 10  chi2=1.4479
start 11  chi2=528.6407
start 12  chi2=4.9203
start 13  chi2=151.4455
start 14  chi2=15.0000
start 15  chi2=5.0888
start 16  chi2=2.1897
start 17  chi2=0.6880
start 18  chi2=3.6047
start 19  chi2=4.9445
start 20  chi2=13.4567
start 21  chi2=0.8685
start 22  chi2=97.4708
start 23  chi2=10253.3487
start 24  chi2=0.4109
start 25  chi2=17.0554
start 26  chi2=18.5377
start 27  chi2=105.7334
start 28  chi2=2.7370
start 29  chi2=14.5751
start 30  chi2=4.6911
start 31  chi2=4.5415
start 32  chi2=6007.2037
start 33  chi2=50.9811
start 34  chi2=3.6044
start 35  chi2=25.4122
start 36  chi2=13.7995
start 37  chi2=13.8267
start 38  chi2=29.8372
start 39  chi2=520.4987
BEST CHI2: 0.4108789563968102
a0         11.907199
phi        -0.470668
sigma     